In [ ]:
import matplotlib.pyplot as plt

from src.data_manager import DataManager
from src.config import Config


config = Config()
data_manager = DataManager(config)

for batch in data_manager:
    spectrogram = batch['input_values'][0]
    spectrogram = spectrogram.T.numpy()
    plt.figure(figsize=(12, 4))
    plt.imshow(spectrogram, aspect='auto', origin='lower', cmap='viridis')
    plt.colorbar(label='Log-Mel energy')
    plt.title('AST Mel spectrogram')
    plt.xlabel('Time frames')
    plt.ylabel('Mel filters (frequency)')
    plt.show()
    break

In [ ]:
import torch
import random

from transformers import ASTForAudioClassification

from src.config import Config
from src.data_manager import DataManager

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
config = Config()
data_manager = DataManager(config)

# Extract random evaluation example.
_, test_dataset = data_manager.get_dataset_splits()
random_idx = random.randint(0, len(test_dataset) - 1)
example = test_dataset[random_idx]

checkpoint_path = "data/checkpoints/checkpoint-20"
model = ASTForAudioClassification.from_pretrained(checkpoint_path)
model.eval()
model.to(device)

with torch.no_grad():
    input = torch.tensor(example['input_values'], device=device).unsqueeze(0)
    logits = model(input).logits
    probs = torch.nn.functional.softmax(logits, dim=-1)


for idx, prob in enumerate(probs):
    predicted_label = data_manager.id_to_label[idx]
    print(idx, prob, predicted_label)

# predicted_label = data_manager.id_to_label[prob_id]
# actual_label = data_manager.id_to_label[example['labels']]
# print(f'predicted={predicted_label}; actual={actual_label}')

Processing 59 rows.


/Users/oscar/Desktop/Projects/birdcleff-2026/venv/lib/python3.11/site-packages/transformers/audio_utils.py:538: UserWarning: At least one mel filter has all zero values. The value for `num_mel_filters` (128) may be set too high. Or, the value for `num_frequency_bins` (257) may be set too low.
  warnings.warn(


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

0 tensor([0.7648, 0.0124, 0.0972, 0.0965, 0.0108, 0.0182], device='mps:0') 22961;23158;24321;517063;65380
